# SQL or Custom Expressions?

SLayer draws a clean line between **SQL** and its **DSL**:

- **Dataset definitions** (dimensions, metrics, filters) use **raw SQL** — they reference the underlying table columns directly.
- **Queries** use the **DSL** — they reference dimension/metric names, apply transforms like `cumsum` and `change`, and filter on named entities.

The dataset is the abstraction boundary between the two. This notebook demonstrates the distinction with working examples.

See also: [SQL vs DSL](sql_vs_dsl.md) | [Expressions](../../concepts/formulas.md) | [Datasets](../../concepts/models.md) | [Queries](../../concepts/queries.md)

**Prerequisites:** `pip install motley-slayer`

In [1]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

engine, storage, datasets = ensure_jaffle_shop()

## Dataset Definitions Use SQL

A dataset's dimensions and metrics define the mapping from semantic names to SQL expressions. The `sql` field is a raw SQL column reference or expression — it's what ends up in the generated `SELECT` and `GROUP BY`.

For example, the `orders` dataset maps the dimension name `order_total` to the SQL column `order_total`, and the metric name `order_total` to an aggregation over `order_total`:

In [2]:
orders_dataset = next(d for d in datasets if d.name == "orders")

print(f"orders dataset has {len(orders_dataset.fields)} fields:")
print()
print("Fields (name -> SQL expression):")
for fld in orders_dataset.fields[:8]:
    sql_str = fld.sql if fld.sql else fld.name
    pk = " [PK]" if fld.primary_key else ""
    print(f"  {fld.name:<20} -> sql: {sql_str!r:<20} datatype: {fld.datatype}{pk}")


orders dataset has 9 fields:

Fields (name -> SQL expression):
  id                   -> sql: 'id'                 datatype: String [PK]
  customer_id          -> sql: 'customer_id'        datatype: String
  ordered_at           -> sql: 'ordered_at'         datatype: Date
  store_id             -> sql: 'store_id'           datatype: String
  subtotal             -> sql: 'subtotal'           datatype: Decimal
  tax_paid             -> sql: 'tax_paid'           datatype: Decimal
  order_total          -> sql: 'order_total'        datatype: Decimal
  status               -> sql: 'status'             datatype: String


Notice that the dimension/metric `name` is a semantic label, while `sql` is the raw SQL expression. They often coincide (e.g., `name: "order_total"`, `sql: "order_total"`), but they don't have to — a dimension could be named `revenue_with_tax` with `sql: "subtotal + tax_paid"`.

Dataset-level filters are also raw SQL:

```yaml
filters:
  - "deleted_at IS NULL"
  - "status <> 'test'"
```

## Queries Use the DSL

When you write a query, you reference **dimension and metric names** — the semantic side. You never write SQL in a query.

Transform functions like `cumsum`, `change`, and `time_shift` are part of the DSL. They compile to complex SQL (CTEs, window functions, self-joins), but you just name them:

In [3]:
result = engine.execute_sync(
    query={
        "population": "orders",
        "time_dimensions": [{"field": "ordered_at", "granularity": "month"}],
        "metrics": [
            "sum(order_total)",
            {"metric": "change(sum(order_total))", "alias": "mom_change"},
        ],
        "order": [{"field": "ordered_at", "direction": "asc"}],
        "limit": 6,
        "offset": 12,
    }
)

print(f"{'Month':<12} {'Revenue':>12} {'MoM Change':>12}")
print("-" * 38)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    rev = row["orders.order_total_sum"]
    chg = row["orders.mom_change"]
    chg_str = f"${chg:+,.0f}" if chg is not None else "-"
    print(f"{month:<12} ${rev:>11,.2f} {chg_str:>12}")

print("\n SQL of the query", result.sql)

Month             Revenue   MoM Change
--------------------------------------
2024-11      $ 112,245.26      $+6,303
2024-12      $ 147,384.14     $+35,139
2025-01      $ 170,012.34     $+22,628
2025-02      $ 157,102.43     $-12,910
2025-03      $ 184,143.21     $+27,041
2025-04      $ 178,340.48      $-5,803

 SQL of the query WITH base AS (
  SELECT
    DATE_TRUNC('MONTH', orders.ordered_at) AS "orders.ordered_at",
    SUM(orders.order_total) AS "orders.order_total_sum"
  FROM orders AS orders
  GROUP BY
    DATE_TRUNC('MONTH', orders.ordered_at)
), shifted__time_shift_inner AS (
  SELECT
    DATE_TRUNC('MONTH', orders.ordered_at) AS "orders.ordered_at",
    SUM(orders.order_total) AS "orders.order_total_sum"
  FROM orders AS orders
  GROUP BY
    DATE_TRUNC('MONTH', orders.ordered_at)
), sjoin__time_shift_inner AS (
  SELECT
    base."orders.ordered_at",
    base."orders.order_total_sum",
    shifted__time_shift_inner."orders.order_total_sum" AS "orders._time_shift_inner"
  FROM ba

The `change()` function compiled to a self-join CTE under the hood — but the query only referenced the metric name `sum(order_total)` and the transform name `change`. No SQL needed to be written by the caller.

## Filters in Queries Reference Dataset Entities

Query-level filters use dimension and metric names — including dotted names for joined dimensions. SLayer resolves them to the correct SQL automatically:

In [4]:
# Filter on a joined dimension name — not a SQL column reference
result = engine.execute_sync(
    query={
        "population": "orders",
        "metrics": ["count(*)", "sum(order_total)"],
        "dimensions": ["stores.name"],
        "filter": ["stores.name in ('Brooklyn', 'Philadelphia')"],
    }
)

for row in result.data:
    print(f"  {row['orders.stores.name']}: {row['orders._count']:,} orders, ${row['orders.order_total_sum']:,.2f}")

print("\nThe filter 'stores.name' resolved to SQL:")
# Show just the WHERE clause from the generated SQL
where_line = [line for line in result.sql.split("\n") if "WHERE" in line or " IN" in line]
for line in where_line:
    print(f"  {line.strip()}")

  Brooklyn: 256,909 orders, $2,805,157.85
  Philadelphia: 197,665 orders, $2,135,270.98

The filter 'stores.name' resolved to SQL:
  WHERE
  stores.name IN ('Brooklyn', 'Philadelphia')


## Row-Level Expressions in Filters

A filter is not limited to names: any row-level expression over the dataset's fields works, written in SLayer's portable function library, so a day-of-week filter needs no extra field:

In [5]:
# date_part is portable query language: ISO weekdays, Saturday = 6, Sunday = 7.
result = engine.execute_sync(
    query={
        "population": "orders",
        "metrics": ["count(*)", "sum(order_total)"],
        "dimensions": ["stores.name"],
        "filter": ["date_part('day_of_week', ordered_at) in (6, 7)"],
        "order": [{"field": "order_total_sum", "direction": "desc"}],
    }
)

print("Weekend orders (row-level expression in the filter):")
for row in result.data:
    print(f"  {row['orders.stores.name']}: {row['orders._count']:,} orders, ${row['orders.order_total_sum']:,.2f}")

Weekend orders (row-level expression in the filter):
  Brooklyn: 91,813 orders, $859,037.14
  Philadelphia: 71,097 orders, $659,777.34
  Chicago: 36,769 orders, $352,259.00
  San Francisco: 32,132 orders, $305,816.57
  New Orleans: 5,354 orders, $49,048.40


SLayer translates `date_part('day_of_week', ordered_at)` into each database's own SQL; raw SQL belongs only in dataset definitions (`DatasetField.sql`, dataset `filters`).

## Query Result as Dataset

What if you want to use DSL transforms like `time_shift` or `change` to define derived metrics or dimensions?

Use `create_dataset()` with a `query` parameter to save a query's result as a permanent dataset. The derived fields become dimensions and metrics on the new dataset, which can then be queried like any other - or just use a query inside another query, as covered in [multistage queries](../06_multistage_queries/multistage_queries.md)

See [Creating Datasets from Queries](../../concepts/models.md#creating-datasets-from-queries) for details.

## Summary

| Layer | Language | Examples |
|-------|----------|----------|
| **Dataset definitions** (dimensions, metrics, filters) | Raw SQL | `sql: "amount"`, `filters: ["deleted_at IS NULL"]` |
| **Queries** (metrics, dimensions, filter) | DSL names | `{"expression": "change(revenue)"}`, `filter: ["status = 'active'"]` |
| **Row-level expressions in queries** | Query language | `filter: ["date_part('day_of_week', ordered_at) in (6, 7)"]` |
| **DSL in datasets** | Query-as-dataset | Save a DSL query result as a permanent dataset |